# Capítulo 1 — Regressão: introdução

**Livro:** começa na página **21** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
set.seed(42)
options(digits = 4, repr.plot.width = 7, repr.plot.height = 4.5)
suppressPackageStartupMessages({ library(glmnet); library(rpart); library(randomForest); library(FNN); library(e1071) })
cat("Pronto.", R.version.string, "
")

## Notação e Suposições · PDF p. 22

#### 🧪 Exercício: Simulando um problema de regressão

Escreva `sim_seno(n, sd)` que gera `x ~ U(0,1)` e `y = sin(2πx) + ruído`. Gere 200 pontos, faça o gráfico de dispersão e desenhe por cima a função verdadeira r(x) = sin(2πx).

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
treino <- sim_seno(200)
plot(treino, pch = 16, col = "grey50")
curve(sin(2 * pi * x), add = TRUE, col = "red", lwd = 2)

## A Função de Risco · PDF p. 27

#### 🧪 Exercício: Risco estimado em dados novos

Com uma amostra **nova** de 5.000 pontos, estime o risco (erro quadrático médio) de dois preditores: g₁(x) = 0 e g₂(x) = sin(2πx). Qual é o menor risco possível aqui? (dica: a variância do ruído)

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
teste <- sim_seno(5000)
c(g1 = mean((teste$y - 0)^2), g2 = mean((teste$y - sin(2 * pi * teste$x))^2), ruido = 0.3^2)

## Seleção de Modelos: Super e Sub-Ajuste · PDF p. 30

#### 🧪 Exercício: Super e sub-ajuste

Ajuste polinômios de grau 1 a 15 em 60 pontos de treino. Calcule o EQM no treino e em 2.000 pontos de teste e desenhe as duas curvas contra o grau. Onde começa o super-ajuste?

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
treino <- sim_seno(60); teste <- sim_seno(2000)
graus <- 1:15
eqm <- sapply(graus, function(g) {
  m <- lm(y ~ poly(x, g), data = treino)
  c(treino = mean(residuals(m)^2), teste = mean((teste$y - predict(m, teste))^2))
})
matplot(graus, t(eqm), type = "b", pch = 16, log = "y", ylab = "EQM", xlab = "grau")
legend("topright", c("treino", "teste"), col = 1:2, lty = 1:2)
graus[which.min(eqm["teste", ])]

### Data Splitting e Validação Cruzada · PDF p. 32

#### 🧪 Exercício: Validação cruzada feita à mão

Sem pacotes, implemente validação cruzada com 5 lotes (*folds*) para escolher o grau do polinômio (1 a 12) em 100 pontos. Compare o grau escolhido com o do exercício anterior.

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
dados <- sim_seno(100)
lote <- sample(rep(1:5, length.out = nrow(dados)))
cv <- sapply(1:12, function(g) mean(sapply(1:5, function(k) {
  m <- lm(y ~ poly(x, g), data = dados[lote != k, ])
  mean((dados$y[lote == k] - predict(m, dados[lote == k, ]))^2)
})))
plot(1:12, cv, type = "b", xlab = "grau", ylab = "EQM (CV)")
which.min(cv)

### Balanço entre Viés e Variância · PDF p. 40

#### 🧪 Exercício: Viés e variância por simulação

Repita 300 vezes: gere 50 pontos, ajuste polinômio de grau 1 e de grau 12 e guarde a previsão em x₀ = 0,25. Calcule viés² e variância de cada modelo (o valor verdadeiro é sin(π/2) = 1).

In [ ]:
sim_seno <- function(n, sd = 0.3) {
  x <- runif(n)
  data.frame(x = x, y = sin(2 * pi * x) + rnorm(n, sd = sd))
}
x0 <- data.frame(x = 0.25)
prev <- replicate(300, {
  d <- sim_seno(50)
  c(grau1 = predict(lm(y ~ x, d), x0), grau12 = predict(lm(y ~ poly(x, 12), d), x0))
})
rbind(vies2 = (rowMeans(prev) - 1)^2, variancia = apply(prev, 1, var))